# Workstream C: complete figure set


Only **nu = 0.05** is used here. 

## 1. Mount and locate the WS-C outputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, json, math
import numpy as np
import pandas as pd

MYDRIVE = "/content/drive/MyDrive"
OUT = None
for c in sorted(glob.glob(os.path.join(MYDRIVE, "**", "WSC"), recursive=True)):
    if os.path.isdir(c) and os.path.exists(os.path.join(c, "wsc_cka_by_slice.csv")):
        OUT = c; break
assert OUT is not None, "WSC folder with wsc_cka_by_slice.csv not found"
print("WSC folder:", OUT)
for f in sorted(os.listdir(OUT)):
    print("   %-46s %12d bytes" % (f, os.path.getsize(os.path.join(OUT, f))))

CKA = pd.read_csv(os.path.join(OUT, "wsc_cka_by_slice.csv"))
print("\ncka rows: %d" % len(CKA))
print("columns :", list(CKA.columns))

NEU = pd.read_csv(os.path.join(OUT, "wsc_neuron_stats.csv"))
print("\nneuron rows: %d" % len(NEU))
print("columns    :", list(NEU.columns))

NU = "0.05"
CKA = CKA[CKA.nu.astype(str) == NU].copy()
NEU = NEU[NEU.nu.astype(str) == NU].copy()
print("\nafter restricting to nu = %s:  cka %d rows, neuron %d rows" % (NU, len(CKA), len(NEU)))
print("PDEs      :", sorted(CKA.pde.unique()))
print("models    :", sorted(CKA.model.unique()))
print("widths    :", sorted(CKA.n_feat.unique()))
print("slices    : %d distinct t values, %.2f to %.2f"
      % (CKA.t.nunique(), CKA.t.min(), CKA.t.max()))
print("seeds     :", sorted(CKA.seed.unique()))

## 2. Plot style and helpers

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9.5, "axes.titlesize": 10.5,
    "axes.labelsize": 9.5, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "legend.fontsize": 8.2, "legend.frameon": True, "legend.framealpha": 0.92,
    "legend.edgecolor": "0.8", "axes.grid": True, "grid.alpha": 0.20,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 120, "savefig.dpi": 300})

C_Q, C_T, C_G = "#14285a", "#c0392b", "#2e8b57"
STYLE = {"qapinn": (C_Q, "QAPINN", "o"), "twin": (C_T, "Classical twin", "s"),
         "gaaf": (C_G, "GAAF-PINN", "^")}
PDENAME = {"burgers": r"Burgers ($\nu=0.05$), principal benchmark",
           "heat": r"Heat ($\alpha=0.10$), smooth-solution control"}

FIGDIR = os.path.join(OUT, "figures")
os.makedirs(FIGDIR, exist_ok=True)
def savefig(fig, name):
    p = os.path.join(FIGDIR, name + ".png")
    fig.savefig(p, bbox_inches="tight"); plt.close(fig); print("[fig]", os.path.basename(p))

FIRST = [c for c in CKA.columns
         if c.startswith("cka_") and c.endswith("_out")
         and ("L0" in c or "quantum" in c or "front" in c)]
print("first-layer CKA columns:", FIRST)

def first_val(df):
    """One first-layer CKA value per row, whichever column is populated."""
    s = pd.Series(np.nan, index=df.index)
    for c in FIRST:
        if c in df: s = s.fillna(df[c])
    return s

LAYER_COLS = sorted([c for c in CKA.columns if c.startswith("cka_") and c.endswith("_out")],
                    key=lambda s: (len(s), s))
print("all layer CKA columns  :", LAYER_COLS)

def seed_note(df):
    """Label a panel honestly when it rests on a single run."""
    k = df.tag.nunique() if "tag" in df else 0
    if k <= 1: return "  [single run, seed 1234]"
    return "  [%d runs]" % k

if "single_seed" in CKA.columns:
    ss = CKA[CKA.single_seed.astype(str).str.lower().isin(["true", "1"])]
    print("\nsingle-seed runs present: %d rows, %d tags"
          % (len(ss), ss.tag.nunique() if len(ss) else 0))
    if len(ss):
        print("  configs:", sorted({(r.pde, r.model, int(r.n_feat))
                                    for r in ss.itertuples()}))

## 3. C1 and C2 : information flow, per benchmark



In [ ]:
for pde in sorted(CKA.pde.unique()):
    D = CKA[CKA.pde == pde].copy(); D["first"] = first_val(D)
    ns = sorted(D.n_feat.unique())
    ncol = 3; nrow = math.ceil(len(ns)/ncol)
    fig, axs = plt.subplots(nrow, ncol, figsize=(4.1*ncol, 3.2*nrow),
                            sharey=True, sharex=True, squeeze=False)
    for i, n in enumerate(ns):
        ax = axs[i//ncol][i % ncol]
        for kind in ["qapinn", "twin", "gaaf"]:
            sub = D[(D.n_feat == n) & (D.model == kind)]
            if sub.empty: continue
            col, lab, _ = STYLE[kind]
            for s, g in sub.groupby("seed"):
                g = g.sort_values("t")
                ax.plot(g.t, g["first"], color=col, alpha=0.55, lw=1.0)
            ax.plot([], [], color=col, label="%s (%d seeds)" % (lab, sub.seed.nunique()))
        ax.set_title("$n=%d$" % n, loc="left"); ax.set_ylim(-0.02, 1.02)
    axs[0][0].legend(loc="lower left", fontsize=7.4)
    for j in range(ncol): axs[nrow-1][j].set_xlabel("$t$")
    for i2 in range(nrow): axs[i2][0].set_ylabel("CKA(first layer, output)")
    for i in range(len(ns), nrow*ncol): axs[i//ncol][i % ncol].axis("off")
    fig.suptitle("WS-C, C1: information surviving from the first layer into the prediction\n%s"
                 % PDENAME[pde], y=1.005)
    plt.tight_layout(); savefig(fig, "C1_cka_first_layer_to_output_%s" % pde)

for pde in sorted(CKA.pde.unique()):
    D = CKA[CKA.pde == pde].copy(); D["first"] = first_val(D)
    fig, ax = plt.subplots(figsize=(7.2, 4.3))
    for kind in ["qapinn", "twin", "gaaf"]:
        sub = D[D.model == kind]
        if sub.empty: continue
        col, lab, mk = STYLE[kind]
        sub = sub[sub["first"].notna()]
        if sub.empty: continue
        g = sub.groupby("n_feat")["first"]
        ax.errorbar(g.mean().index, g.mean().values, yerr=g.std().values,
                    color=col, marker=mk, ms=5, lw=1.6, capsize=3,
                    label="%s (%d runs)" % (lab, sub.tag.nunique()))
    ax.set_xlabel("qubit count / feature width $n$")
    ax.set_ylabel("CKA(first layer, output)")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title("WS-C, C2: does the first layer matter more as it widens?\n%s" % PDENAME[pde],
                 loc="left")
    ax.legend(loc="best"); savefig(fig, "C2_cka_vs_width_%s" % pde)

## 4. C3 : where the output representation is formed, per benchmark

In [ ]:
for pde in sorted(CKA.pde.unique()):
    D = CKA[CKA.pde == pde]
    fig, ax = plt.subplots(figsize=(7.6, 4.3))
    for kind in ["qapinn", "twin", "gaaf"]:
        sub = D[D.model == kind]
        if sub.empty: continue
        col, lab, mk = STYLE[kind]
        cols = [c for c in LAYER_COLS if sub[c].notna().any()]
        mu = [sub[c].mean() for c in cols]
        sd = [sub[c].std() for c in cols]
        ax.errorbar(range(len(cols)), mu, yerr=sd, color=col, marker=mk, ms=5,
                    lw=1.6, capsize=2.5, label=lab)
        ax.set_xticks(range(len(cols)))
        ax.set_xticklabels([c[4:-4] for c in cols], rotation=45, fontsize=8)
    ax.set_ylabel("CKA(layer, output)"); ax.set_ylim(-0.02, 1.05)
    ax.set_xlabel("layer (first layer on the left, last hidden layer on the right)")
    ax.set_title("WS-C, C3: where the output representation is formed\n%s" % PDENAME[pde],
                 loc="left")
    ax.legend(loc="best"); savefig(fig, "C3_cka_by_depth_%s" % pde)

## 5. C4 : the Figure-3 reproduction, at every width



In [ ]:
T_REF = float(sorted(NEU.t.unique())[len(sorted(NEU.t.unique()))//2])
print("reference slice for C4: t = %.2f  (all 26 slices are used in C6)" % T_REF)
SEED = 1234 if 1234 in set(NEU.seed.unique()) else sorted(NEU.seed.unique())[0]
print("reference seed: %d" % SEED)

for pde in sorted(NEU.pde.unique()):
    for n in sorted(NEU[NEU.pde == pde].n_feat.unique()):
        sub = NEU[(NEU.pde == pde) & (NEU.n_feat == n) & (NEU.seed == SEED)
                  & (np.isclose(NEU.t, T_REF)) & (NEU.layer.isin(["L2", "L3"]))]
        if sub.empty:
            print("[skip] C4 %s n=%d: no rows" % (pde, n)); continue
        kinds = [k for k in ["qapinn", "twin", "gaaf"] if k in set(sub.model.unique())]
        fig, axs = plt.subplots(2, len(kinds), figsize=(4.2*len(kinds), 6.0), squeeze=False,
                                sharey="row")
        for c_, kind in enumerate(kinds):
            col, lab, _ = STYLE[kind]
            for r_, layer in enumerate(["L2", "L3"]):
                ax = axs[r_][c_]
                g = sub[(sub.model == kind) & (sub.layer == layer)].sort_values("neuron")
                ax.bar(g.neuron, g["std"], color=col, alpha=0.88)
                ax.set_title("%s, layer %s" % (lab, layer), loc="left", fontsize=9.5)
                ax.set_xlabel("neuron index")
                if c_ == 0: ax.set_ylabel("activation standard deviation")
        fig.suptitle("WS-C, C4: per-neuron activation spread, $n=%d$, seed %d, $t=%.2f$\n%s%s"
                     % (n, SEED, T_REF, PDENAME[pde], seed_note(sub)), y=1.01)
        plt.tight_layout(); savefig(fig, "C4_figure3_%s_n%d" % (pde, n))

## 6. C6 : every neuron against every time slice



In [ ]:
for pde in sorted(NEU.pde.unique()):
    for n in sorted(NEU[NEU.pde == pde].n_feat.unique()):
        base = NEU[(NEU.pde == pde) & (NEU.n_feat == n) & (NEU.seed == SEED)]
        if base.empty: continue
        kinds = [k for k in ["qapinn", "twin", "gaaf"] if k in set(base.model.unique())]
        # the first layer is included: it is the one that differs between architectures
        layers = ["FIRST", "L2", "L3"]
        fig, axs = plt.subplots(len(layers), len(kinds),
                                figsize=(4.3*len(kinds), 3.1*len(layers)), squeeze=False)
        _v = base[base.layer.isin(["L2", "L3"])]["std"].quantile(0.99)
        vmax = float(_v) if np.isfinite(_v) else 1.0
        for c_, kind in enumerate(kinds):
            for r_, layer in enumerate(layers):
                ax = axs[r_][c_]
                if layer == "FIRST":
                    g = base[(base.model == kind) & (base.layer.str.startswith("L0"))]
                    lname = g.layer.iloc[0] if not g.empty else "L0"
                    _vv = g["std"].quantile(0.99) if not g.empty else np.nan
                    vm = float(_vv) if np.isfinite(_vv) else vmax
                else:
                    g = base[(base.model == kind) & (base.layer == layer)]
                    lname, vm = layer, vmax
                if g.empty or not np.isfinite(vm) or vm <= 0:
                    ax.axis("off"); continue
                M = g.pivot_table(index="neuron", columns="t", values="std")
                im = ax.imshow(M.values, aspect="auto", origin="lower", cmap="magma",
                               vmin=0, vmax=vm,
                               extent=[M.columns.min(), M.columns.max(), -0.5,
                                       M.index.max()+0.5])
                ax.set_title("%s, %s" % (STYLE[kind][1], lname), loc="left", fontsize=9.5)
                if r_ == len(layers) - 1: ax.set_xlabel("$t$")
                if c_ == 0: ax.set_ylabel("neuron index")
                ax.grid(False)
                if layer == "FIRST":
                    fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
        fig.colorbar(im, ax=axs[1:, :].ravel().tolist(), fraction=0.02,
                     label="activation standard deviation (rows 2-3)")
        fig.suptitle("WS-C, C6: every neuron across all %d time slices, $n=%d$, seed %d\n"
                     "top row is the layer that differs between architectures\n%s%s"
                     % (NEU.t.nunique(), n, SEED, PDENAME[pde], seed_note(base)), y=1.01)
        savefig(fig, "C6_neuron_time_%s_n%d" % (pde, n))

## 7. C7 : which neurons actually drive the output



In [ ]:
for pde in sorted(NEU.pde.unique()):
    D = NEU[NEU.pde == pde]
    layers = sorted(D.layer.unique(), key=lambda s: (len(s), s))
    fig, ax = plt.subplots(figsize=(8.0, 4.3))
    w = 0.26
    for k_, kind in enumerate(["qapinn", "twin", "gaaf"]):
        sub = D[D.model == kind]
        if sub.empty: continue
        col, lab, _ = STYLE[kind]
        present = [L for L in layers if L in set(sub.layer.unique())]
        xs = np.arange(len(present)) + (k_ - 1)*w
        mu = [sub[sub.layer == L].cka_out.mean() for L in present]
        sd = [sub[sub.layer == L].cka_out.std() for L in present]
        ax.bar(xs, mu, w, yerr=sd, color=col, alpha=0.9, capsize=2, label=lab)
    ax.set_xticks(np.arange(len(layers))); ax.set_xticklabels(layers, rotation=45, fontsize=8)
    ax.set_ylabel("mean per-neuron CKA with the output")
    ax.set_title("WS-C, C7: individual neuron contribution to the prediction, by layer\n%s"
                 % PDENAME[pde], loc="left")
    ax.legend(loc="best"); savefig(fig, "C7_neuron_cka_by_layer_%s" % pde)

# distribution of per-neuron contribution in the first layer, where the models differ
for pde in sorted(NEU.pde.unique()):
    D = NEU[(NEU.pde == pde) & (NEU.layer.str.startswith("L0"))]
    if D.empty: continue
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    for kind in ["qapinn", "twin", "gaaf"]:
        sub = D[D.model == kind]
        if sub.empty: continue
        col, lab, _ = STYLE[kind]
        vals = sub.cka_out.dropna()
        if len(vals) < 5: continue
        ax.hist(vals, bins=40, histtype="step", lw=1.8,
                color=col, density=True, label="%s (%d values)" % (lab, len(vals)))
    ax.set_xlabel("per-neuron CKA with the output, first layer")
    ax.set_ylabel("density")
    ax.set_title("WS-C, C7b: distribution of first-layer neuron contributions\n%s"
                 % PDENAME[pde], loc="left")
    ax.legend(loc="best"); savefig(fig, "C7b_first_layer_cka_hist_%s" % pde)

## 8. C5 : neuron saturation, per benchmark

In [ ]:
for pde in sorted(NEU.pde.unique()):
    D = NEU[NEU.pde == pde]
    fig, ax = plt.subplots(figsize=(7.6, 4.3))
    for kind in ["qapinn", "twin", "gaaf"]:
        sub = D[D.model == kind]
        if sub.empty: continue
        col, lab, mk = STYLE[kind]
        layers = sorted(sub.layer.unique(), key=lambda s: (len(s), s))
        mu = [sub[sub.layer == L].sat.mean() for L in layers]
        ax.plot(range(len(layers)), mu, color=col, marker=mk, ms=5, lw=1.6, label=lab)
        ax.set_xticks(range(len(layers))); ax.set_xticklabels(layers, rotation=45, fontsize=8)
    ax.set_ylabel(r"fraction of inputs with $|a|>0.95$")
    ax.set_xlabel("layer")
    ax.set_title("WS-C, C5: neuron saturation by layer\n%s" % PDENAME[pde], loc="left")
    ax.legend(loc="best"); savefig(fig, "C5_saturation_by_layer_%s" % pde)

## 9. Summary table and manifest

In [ ]:
rows = []
for pde in sorted(CKA.pde.unique()):
    D = CKA[CKA.pde == pde].copy(); D["first"] = first_val(D)
    for kind in sorted(D.model.unique()):
        for n in sorted(D[D.model == kind].n_feat.unique()):
            s = D[(D.model == kind) & (D.n_feat == n)]
            rows.append(dict(pde=pde, model=kind, n_feat=int(n),
                             n_runs=int(s.tag.nunique()), n_slices=int(s.t.nunique()),
                             cka_first_mean=float(s["first"].mean()),
                             cka_first_sd=float(s["first"].std()),
                             cka_first_min=float(s["first"].min()),
                             cka_first_max=float(s["first"].max())))
S = pd.DataFrame(rows)
S.to_csv(os.path.join(OUT, "wsc_summary_by_config.csv"), index=False)
print(S.to_string(index=False))
print("\n[saved] wsc_summary_by_config.csv")

figs = sorted(os.listdir(FIGDIR))
man = dict(note="wsc-figures (Track A, Djabon), regenerated from the stored CSVs",
           nu=NU, source_csvs=["wsc_cka_by_slice.csv", "wsc_neuron_stats.csv"],
           n_cka_rows=int(len(CKA)), n_neuron_rows=int(len(NEU)),
           pdes=sorted(CKA.pde.unique().tolist()),
           widths=sorted(int(x) for x in CKA.n_feat.unique()),
           n_slices=int(CKA.t.nunique()),
           reference_slice_C4=T_REF, reference_seed=int(SEED),
           figures=figs,
           caveats=[
               "Every figure is produced separately per PDE; no figure pools benchmarks.",
               "C4 uses one reference slice so panels are comparable; C6 uses all slices.",
               "Only nu=0.05 is included; hard-viscosity runs are analysed separately.",
               "Nothing was recomputed: all values come from the stored CSVs.",
           ])
json.dump(man, open(os.path.join(OUT, "wsc_figures_manifest.json"), "w"), indent=2)
print("\n%d figures in %s" % (len(figs), FIGDIR))
for f in figs: print("   ", f)